In [1]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation


C:\Users\pkell\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [130]:
## Split Half Group level

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\1st_half_contrast_ROI_AGG_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\2nd_half_contrast_ROI_AGG_clean.csv"

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\1st_half_GLM_GROUP_ROI_AGG_tvr_clean.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\2nd_half_GLM_GROUP_ROI_AGG_tvr_clean.csv"

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_contrast_ROI_AGG_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_contrast_ROI_AGG_clean.csv"

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_blocks_GLM_GROUP_ROI_AGG_tvr_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_blocks_GLM_GROUP_ROI_AGG_tvr_clean.csv"

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_contrast_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_contrast_clean.csv"

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_blocks_GLM_GROUP_tvr_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_blocks_GLM_GROUP_tvr_clean.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# # --- Drop Short Separation Channels ---
# if 'ShortSeperation' in df1.columns:
#     df1 = df1[df1['ShortSeperation'] != 1]
# if 'ShortSeperation' in df2.columns:
#     df2 = df2[df2['ShortSeperation'] != 1]

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['detector'].isin(bad_detectors)]
df2 = df2[~df2['detector'].isin(bad_detectors)]

# # --- Drop Specific Source-Detector Pairs ---
df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
df1 = df1[~((df1['source'] == 16) & (df1['detector'] == 12))]

df2 = df2[~((df2['source'] == 14) & (df2['detector'] == 13))]
df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Filter by q-value threshold ---
df1 = df1[df1['q'] <= 0.06]
df2 = df2[df2['q'] <= 0.06]


# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Combine Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Initialize Results ---
condition_results = []

# --- Run Split-Half Analysis Per Condition ---
for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]

    session1 = cond_data[cond_data['session'] == 1]
    session2 = cond_data[cond_data['session'] == 2]

    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for condition {condition}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        sb_corrected = (2 * r) / (1 + r) if r < 1 else 1.0
        var1 = merged['tstat_s1'].var()
        var2 = merged['tstat_s2'].var()
        var_stability = min(var1, var2) / max(var1, var2) if max(var1, var2) > 0 else 0

        condition_results.append({
            'condition': condition,
            'pearson_r': r,
            'spearman_brown': sb_corrected,
            'variance_stability': var_stability,
            'n_channels': merged.shape[0],
            'p_value': p
        })
    except Exception as e:
        print(f"❌ Error computing r for condition {condition}: {e}")

# --- Convert to DataFrame ---
group_corr_df = pd.DataFrame(condition_results)
print(group_corr_df)

# --- Reshape for Grouped Bar Plot ---
plot_df = pd.melt(
    group_corr_df,
    id_vars='condition',
    value_vars=['pearson_r', 'spearman_brown', 'variance_stability'],
    var_name='Metric',
    value_name='Value'
)

# --- Create Grouped Bar Plot ---
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='condition', y='Value', hue='Metric', palette='pastel', edgecolor='black')

# Add value annotations to each bar
for container in plt.gca().containers:
    for bar in container:
        height = bar.get_height()
        if not pd.isna(height):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.02,
                f'{height:.2f}',
                ha='center',
                va='bottom',
                fontsize=9
            )

plt.title('Split-Half Metrics by Condition')
plt.xlabel('Condition')
plt.ylabel('Metric Value')
plt.ylim(0, 1.2)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()



  condition  pearson_r  spearman_brown  variance_stability  n_channels  \
0        x1   0.494729        0.661965             0.07517           5   

    p_value  
0  0.396824  


In [129]:
## Split Half Subject Level 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Subject-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_subject_level_contrasts.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_subject_level_contrasts.csv"

# # --- Load Subject-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\even_subject_level_contrasts_ROI_AGG.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\odd_subject_level_contrasts_ROI_AGG.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['detector'].isin(bad_detectors)]
df2 = df2[~df2['detector'].isin(bad_detectors)]

# --- Filter by q-value threshold ---
df1 = df1[df1['q'] <= 0.06]
df2 = df2[df2['q'] <= 0.06]

# --- Label halves ---
df1['half'] = 'even'
df2['half'] = 'odd'

# --- Combine ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Per Subject/Condition Analysis ---
subject_results = []

for (subj, cond), group in combined.groupby(['subject', 'cond']):
    even = group[group['half'] == 'even']
    odd = group[group['half'] == 'odd']

    merged = pd.merge(
        even, odd,
        on=['source', 'detector'],
        suffixes=('_even', '_odd')
    )

    if merged.shape[0] < 3:
        continue

    try:
        # Correlation
        r, _ = pearsonr(merged['tstat_even'], merged['tstat_odd'])
        sb_corr = (2 * r) / (1 + r) if r < 1 else 1.0

        # Variance Stability = min(var1, var2) / max(var1, var2)
        var_even = merged['tstat_even'].var()
        var_odd = merged['tstat_odd'].var()
        if var_even > 0 and var_odd > 0:
            var_stability = min(var_even, var_odd) / max(var_even, var_odd)
        else:
            var_stability = 0  # degenerate case

        subject_results.append({
            'subject': subj,
            'condition': cond,
            'pearson_r': r,
            'spearman_brown': sb_corr,
            'variance_stability': var_stability
        })
    except:
        continue

# --- Convert to DataFrame ---
subject_df = pd.DataFrame(subject_results)

# --- Summary Stats ---
summary_df = subject_df.groupby('condition').agg(
    mean_pearson_r=('pearson_r', 'mean'),
    mean_spearman_brown=('spearman_brown', 'mean'),
    mean_variance_stability=('variance_stability', 'mean'),
    n=('subject', 'count')
).reset_index()

print(summary_df)
print(subject_df)


# --- Plot Pearson r ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='pearson_r', hue='subject', dodge=True)
plt.title('Subject-Level Split-Half Pearson r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Pearson r')
plt.ylim(0, 1.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Plot Spearman-Brown corrected r ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='spearman_brown', hue='subject', dodge=True)
plt.title('Subject-Level Spearman-Brown Corrected r (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Spearman-Brown Corrected r')
plt.ylim(0, 1.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Plot Variance Stability ---
plt.figure(figsize=(10, 5))
sns.barplot(data=subject_df, x='condition', y='variance_stability', hue='subject', dodge=True)
plt.title('Subject-Level Variance Stability (Odd vs Even Blocks)')
plt.xlabel('Condition')
plt.ylabel('Variance Stability (min/max)')
plt.ylim(0, 1.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
plt.tight_layout()
plt.show()

# --- Mean Plot: Pearson r, SB, and Variance Stability ---
plot_df = pd.melt(
    subject_df,
    id_vars=['condition'],
    value_vars=['pearson_r', 'spearman_brown', 'variance_stability'],
    var_name='Metric',
    value_name='r_value'
)

mean_df = plot_df.groupby(['condition', 'Metric'], as_index=False).mean()

plt.figure(figsize=(8, 5))
sns.barplot(data=mean_df, x='condition', y='r_value', hue='Metric', palette='pastel', edgecolor='black')

for i, row in mean_df.iterrows():
    plt.text(
        x=i % len(mean_df['condition'].unique()) + (0.15 if row['Metric'] != 'pearson_r' else -0.15),
        y=row['r_value'] + 0.02,
        s=f"{row['r_value']:.2f}",
        ha='center', va='bottom', fontsize=10
    )

plt.title('Mean Split-Half Metrics by Condition')
plt.ylabel('Mean Value')
plt.xlabel('Condition')
plt.ylim(0, 1.2)
plt.legend(title='Metric')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


KeyError: 'condition'

In [142]:
## Group Level Task v Rest Correlation
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, ttest_rel
import matplotlib.pyplot as plt
import seaborn as sns

# # --- Load Group-Level Channel Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_session1_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_session2_clean.csv"

# # --- Load Group-Level Channel Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_session1_tvr_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_session2_tvr_clean.csv"

# # --- Load Group-Level ROI Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_ROI_AGG_session1_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_ROI_AGG_session2_clean.csv"

# --- Load Group-Level ROI Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_ROI_AGG_session1_tvr_clean.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_ROI_AGG_session2_tvr_clean.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# # --- Drop Rows with Detector in {16–23} ---
# bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
# df1 = df1[~df1['detector'].isin(bad_detectors)]
# df2 = df2[~df2['detector'].isin(bad_detectors)]

# # --- Drop Specific Source-Detector Pairs ---
# df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
# df1 = df1[~((df1['source'] == 16) & (df1['detector'] == 12))]

# df2 = df2[~((df2['source'] == 14) & (df2['detector'] == 13))]
# df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Match channels by source and detector across sessions ---
merged = pd.merge(
    df1,
    df2,
    on=['source', 'detector'],
    suffixes=('_s1', '_s2')
)

if merged.shape[0] < 3:
    print("⚠️ Not enough matched channels between sessions. Exiting.")
else:
    # Compute Pearson correlation between t-values
    r, p_corr = pearsonr(merged['tstat_s1'], merged['tstat_s2'])

    # Compute paired t-test between session 1 and 2 t-values
    t_stat, p_ttest = ttest_rel(merged['tstat_s1'], merged['tstat_s2'])

    print(f"✅ Pearson r (Session 1 vs. Session 2 t-values): {r:.2f}, p-value: {p_corr:.2g}")
    print(f"Matched channels: {merged.shape[0]}")
    print(f"🔍 Paired t-test on t-values: t = {t_stat:.2f}, p = {p_ttest:.2g}")

    # --- Plotting ---
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=merged['tstat_s1'], y=merged['tstat_s2'], s=60, color='steelblue', edgecolor='black')
    
    # Line of best fit
    fit = np.polyfit(merged['tstat_s1'], merged['tstat_s2'], 1)
    fit_line = np.poly1d(fit)
    x_vals = np.linspace(merged['tstat_s1'].min(), merged['tstat_s1'].max(), 100)
    plt.plot(x_vals, fit_line(x_vals), linestyle='--', color='black')

    # Add correlation stats to plot
    plt.text(0.05, 0.95,
             f'r = {r:.2f}\np = {p_corr:.2g}',
             transform=plt.gca().transAxes,
             verticalalignment='top',
             fontsize=11,
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title('Activation Stability: Task vs Baseline', fontsize=18)
    plt.xlabel('Session 1 t values', fontsize=16)
    plt.ylabel('Session 2 t values', fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    #  # --- Save figure ---
    # save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_group_correlation_roi_tvr.png"  # <-- EDIT THIS PATH
    # plt.savefig(save_path, dpi=300)
    # print(f"📁 Figure saved as: {save_path}")

    plt.show()


✅ Pearson r (Session 1 vs. Session 2 t-values): 0.87, p-value: 0.00097
Matched channels: 10
🔍 Paired t-test on t-values: t = 4.53, p = 0.0014
📁 Figure saved as: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_group_correlation_roi_tvr.png


In [140]:
## Group Level Task v Rest Correlation - Variance 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_session1_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_session2_clean.csv"

# # --- Load Group-Level Channel Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_session1_tvr_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_session2_tvr_clean.csv"

# --- Load Group-Level ROI Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_ROI_AGG_session1_clean.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Contrast_glm_result_ROI_AGG_session2_clean.csv"

# # --- Load Group-Level ROI Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_ROI_AGG_session1_tvr_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_Group_glm_result_ROI_AGG_session2_tvr_clean.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# # --- Drop Rows with Detector in {16–23} ---
# bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
# df1 = df1[~df1['detector'].isin(bad_detectors)]
# df2 = df2[~df2['detector'].isin(bad_detectors)]

# # --- Drop Specific Source-Detector Pairs ---
# df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
# df1 = df1[~((df1['source'] == 16) & (df1['detector'] == 12))]

# df2 = df2[~((df2['source'] == 14) & (df2['detector'] == 13))]
# df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Match channels by source and detector across sessions ---
merged = pd.merge(
    df1,
    df2,
    on=['source', 'detector'],
    suffixes=('_s1', '_s2')
)

if merged.shape[0] < 3:
    print("⚠️ Not enough matched channels between sessions. Exiting.")
else:
    # Compute squared t-values (proxy for variance)
    var_s1 = merged['beta_s1'] ** 2
    var_s2 = merged['beta_s2'] ** 2

    # Correlation between variances
    r, p = pearsonr(var_s1, var_s2)

    print(f"✅ Pearson r (Session 1 vs. Session 2 variance): {r:.4f}, p-value: {p:.4g}")
    print(f"Matched channels: {merged.shape[0]}")

    # --- Plotting ---
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=var_s1, y=var_s2, s=60, color='mediumseagreen', edgecolor='black')

    # Identity line
    plt.plot([var_s1.min(), var_s1.max()],
             [var_s1.min(), var_s1.max()],
             linestyle='--', color='gray')

    # Add r and p-value annotation
    plt.text(0.05, 0.95,
             f'r = {r:.2f}\np = {p:.4g}',
             transform=plt.gca().transAxes,
             verticalalignment='top',
             fontsize=11,
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.title('Variance Stability: 2nd vs 1st')
    plt.xlabel('Session 1 Variance')
    plt.ylabel('Session 2 Variance')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    #  # --- Save figure ---
    # save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_group_correlation_variance_roi.png"  # <-- EDIT THIS PATH
    # plt.savefig(save_path, dpi=300)
    # print(f"📁 Figure saved as: {save_path}")
    
    plt.show()


✅ Pearson r (Session 1 vs. Session 2 variance): 0.3085, p-value: 0.3858
Matched channels: 10
📁 Figure saved as: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_group_correlation_variance_roi.png


In [50]:
## Participant Session to Session Correlations - TASK VS REST  

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- Load Channel Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session2.csv"

# # --- Load ROI Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_ROI_AGG_session1.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_ROI_AGG_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop unnecessary columns if they exist ---
df1 = df1.drop(columns=['contact_info', 'remarks'], errors='ignore')
df2 = df2.drop(columns=['contact_info', 'remarks'], errors='ignore')

# --- Normalize subject names by keeping only the last 7 characters ---
df1['subject'] = df1['subject'].astype(str).str[-7:]
df2['subject'] = df2['subject'].astype(str).str[-7:]

# --- Drop only rows with NaNs ---
df1 = df1.dropna(how='any')
df2 = df2.dropna(how='any')


# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Each Participant Across Channels ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]
    
    session1 = subj_data[subj_data['session'] == 1].copy()
    session2 = subj_data[subj_data['session'] == 2].copy()

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector', 'cond'],  # Match on channel and condition
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough channels to compute correlation for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['tstat_s1'], merged['tstat_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_channels': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)

# Calculate summary statistics
min_r = np.min(subject_corr_df['pearson_r'])
max_r = np.max(subject_corr_df['pearson_r'])
mean_r = np.mean(subject_corr_df['pearson_r'])
std_r = np.std(subject_corr_df['pearson_r'], ddof=1)  # sample standard deviation

# Print the results
print(f"Min Pearson r: {min_r:.4f}")
print(f"Max Pearson r: {max_r:.4f}")
print(f"Mean Pearson r: {mean_r:.4f}")
print(f"Standard Deviation of Pearson r: {std_r:.4f}")

import matplotlib.pyplot as plt
import seaborn as sns

# --- Bar Plot of Session-to-Session r for Each Participant ---
plt.figure(figsize=(12, 6))
sns.set(style="whitegrid", font_scale=1.2)

# Create barplot
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='Blues_d')

# Add r value text above each bar
for index, row in subject_corr_df.iterrows():
    ax.text(index, row['pearson_r'] + 0.01, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=10)

# Final plot formatting
plt.xticks(rotation=45, ha='right')
plt.ylabel("Pearson r (Session 1 vs Session 2)")
plt.xlabel("Participant")
plt.title("Participant Level Test-Retest Correlation (ROI)")
plt.ylim(-.6, 1)  # Adjust as needed based on r value range
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.
    subject  pearson_r  n_channels
0   pid001\   0.059566          46
1   pid002\   0.045539          46
2   pid003\  -0.181728          46
3   pid004\  -0.371654          46
4   pid005\  -0.270236          46
5   pid006\   0.339602          46
6   pid007\   0.593901          46
7   pid008\   0.334400          46
8   pid009\   0.383513          46
9   pid010\  -0.138215          46
10  pid011\   0.055153          46
11  pid012\   0.063660          46
12  pid013\   0.156052          46
13  pid014\   0.264347          46
14  pid015\  -0.247773          46
15  pid016\   0.312761          46
16  pid017\   0.038367          46
17  pid018\  -0.258413          46
18  pid019\   0.304594          46
19  pid020\   0.258309          46
20  pid021\   0.511580          46
21  pid022\   0.375452          46
22  pid023\   0.257819          46
23  pid024\   0.086789          46
24  pid025\   0.236888          46
25  pid026\   0.414367          46
Min Pearson r: -0

C:\Users\pkell\AppData\Local\Temp\ipykernel_51832\2341361006.py:105: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='Blues_d')


In [48]:
## Subject Level Variance Correlation Task vs Rest 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- Load Channel Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session2.csv"

# # --- Load ROI Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_ROI_AGG_session1.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_ROI_AGG_session2.csv"


df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop unnecessary columns if they exist ---
df1 = df1.drop(columns=['contact_info', 'remarks'], errors='ignore')
df2 = df2.drop(columns=['contact_info', 'remarks'], errors='ignore')

# --- Normalize subject names by keeping only the last 7 characters ---
df1['subject'] = df1['subject'].astype(str).str[-7:]
df2['subject'] = df2['subject'].astype(str).str[-7:]

# --- Drop only rows with NaNs ---
df1 = df1.dropna(how='any')
df2 = df2.dropna(how='any')

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r on t² for Each Participant Across Channels ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]
    
    session1 = subj_data[subj_data['session'] == 1].copy()
    session2 = subj_data[subj_data['session'] == 2].copy()

    # Match on channel and condition
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector', 'cond'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    try:
        t2_s1 = merged['tstat_s1']**2
        t2_s2 = merged['tstat_s2']**2
        r, p = pearsonr(t2_s1, t2_s2)
        participant_results.append({
            'subject': subject,
            'pearson_r_variance': r,
            'n_channels': len(t2_s1)
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
var_stab_df = pd.DataFrame(participant_results)
print(var_stab_df)

# Calculate summary statistics
min_r = np.min(var_stab_df['pearson_r_variance'])
max_r = np.max(var_stab_df['pearson_r_variance'])
mean_r = np.mean(var_stab_df['pearson_r_variance'])
std_r = np.std(var_stab_df['pearson_r_variance'], ddof=1)  # sample standard deviation

# Print the results
print(f"Min Pearson r: {min_r:.4f}")
print(f"Max Pearson r: {max_r:.4f}")
print(f"Mean Pearson r: {mean_r:.4f}")
print(f"Standard Deviation of Pearson r: {std_r:.4f}")

# --- Plot Histogram of Variance r Values ---
plt.figure(figsize=(8, 5))
plt.hist(var_stab_df['pearson_r_variance'], bins=10, edgecolor='black', color='teal')
plt.title('Participant-Level Variance Stability (t² Correlation)')
plt.xlabel('Pearson r (t² Session 1 vs Session 2)')
plt.ylabel('Number of Participants')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- Bar Plot of Variance Stability per Participant ---
import seaborn as sns

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=var_stab_df, x='subject', y='pearson_r_variance', palette='coolwarm', edgecolor='black')

# Add text labels above each bar
for i, row in var_stab_df.iterrows():
    ax.text(
        i,
        row['pearson_r_variance'] + 0.02,
        f"{row['pearson_r_variance']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Variance Stability (ROI)', fontsize=14)
plt.xlabel('Participant')
plt.ylabel('Pearson r (t² Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(-0.6, 1.1)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()



✅ Found 26 subjects in both sessions.
    subject  pearson_r_variance  n_channels
0   pid001\           -0.104778          46
1   pid002\            0.192857          46
2   pid003\           -0.064318          46
3   pid004\           -0.057877          46
4   pid005\            0.268458          46
5   pid006\           -0.294161          46
6   pid007\            0.345918          46
7   pid008\            0.145148          46
8   pid009\            0.135182          46
9   pid010\           -0.096616          46
10  pid011\           -0.115836          46
11  pid012\           -0.040397          46
12  pid013\            0.074380          46
13  pid014\            0.118587          46
14  pid015\            0.140213          46
15  pid016\           -0.361692          46
16  pid017\           -0.053425          46
17  pid018\            0.151701          46
18  pid019\            0.083038          46
19  pid020\            0.097128          46
20  pid021\            0.308961       

C:\Users\pkell\AppData\Local\Temp\ipykernel_51832\2816328639.py:113: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=var_stab_df, x='subject', y='pearson_r_variance', palette='coolwarm', edgecolor='black')


In [52]:
## Subject Level Task vs Rest Correlation Permutation

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# # --- Load Channel Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session1.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session2.csv"

# --- Load ROI Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_ROI_AGG_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_ROI_AGG_session2.csv"


df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Drop rows with NaN in 'tstat' column ---
df1 = df1.dropna(subset=['tstat'])
df2 = df2.dropna(subset=['tstat'])

# --- Normalize subject names by keeping only the last 7 characters ---
df1['subject'] = df1['subject'].astype(str).str[-7:]
df2['subject'] = df2['subject'].astype(str).str[-7:]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Permutation Parameters ---
n_iter = 1000
permutation_results = []

# --- Run Permutation Test Per Subject ---
for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Match channels by source and detector
    merged = pd.merge(
        session1,
        session2,
        on=['source', 'detector'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched channels for {subject}. Skipping.")
        continue

    # True r
    s1_vec = merged['tstat_s1'].values
    s2_vec = merged['tstat_s2'].values
    true_r, _ = pearsonr(s1_vec, s2_vec)
    perm_r_vals = []

    for _ in range(n_iter):
        s2_perm = np.random.permutation(s2_vec)
        r, _ = pearsonr(s1_vec, s2_perm)
        perm_r_vals.append(r)

    permutation_results.append({
        'subject': subject,
        'true_r': true_r,
        'perm_r_vals': perm_r_vals,
        'p_value': np.mean(np.array(perm_r_vals) >= true_r)
    })

# --- Create DataFrame ---
perm_df = pd.DataFrame(permutation_results)

# --- Plotting ---
plt.figure(figsize=(12, 6))

for _, row in perm_df.iterrows():
    sns.kdeplot(row['perm_r_vals'], fill=True, alpha=0.2, label=f"{row['subject']} permuted")
    plt.axvline(row['true_r'], linestyle='--', linewidth=2, alpha=0.8, label=f"{row['subject']} true r")

plt.title("Within-Subject Permutation Distributions vs. True r Values Task vs Rest")
plt.xlabel("Pearson r")
plt.ylabel("Density")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# --- Print summary table ---
print(perm_df[['subject', 'true_r', 'p_value']])


✅ Found 26 subjects in both sessions.
    subject    true_r  p_value
0   pid001\ -0.549504    0.955
1   pid002\  0.139013    0.375
2   pid003\ -0.211050    0.740
3   pid004\ -0.246280    0.719
4   pid005\ -0.508571    0.939
5   pid006\  0.228373    0.241
6   pid007\  0.935440    0.000
7   pid008\  0.443444    0.096
8   pid009\  0.434360    0.114
9   pid010\ -0.357580    0.847
10  pid011\  0.520910    0.112
11  pid012\  0.599324    0.020
12  pid013\ -0.060078    0.587
13  pid014\  0.100094    0.402
14  pid015\ -0.743900    0.994
15  pid016\  0.390879    0.145
16  pid017\  0.434892    0.105
17  pid018\ -0.128675    0.651
18  pid019\  0.381760    0.141
19  pid020\  0.299686    0.198
20  pid021\  0.389018    0.132
21  pid022\  0.300027    0.206
22  pid023\  0.341167    0.177
23  pid024\ -0.248999    0.749
24  pid025\ -0.115790    0.629
25  pid026\  0.613350    0.025


In [111]:
# TASK VS REST - Channel Level Correlation Code - SUBJECT LEVEL 
import pandas as pd
from scipy.stats import pearsonr

# # --- Load Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_glm_result_session1_tvr_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_glm_result_session2_tvr_clean.csv"

## --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session1_clean.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session2_clean.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Normalize subject names by keeping only the last 7 characters ---
df1['subject'] = df1['subject'].astype(str).str[-7:]
df2['subject'] = df2['subject'].astype(str).str[-7:]

# --- Drop rows with NaN in 'tstat' column ---
df1 = df1.dropna(subset=['beta'])
df2 = df2.dropna(subset=['beta'])

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['detector'].isin(bad_detectors)]
df2 = df2[~df2['detector'].isin(bad_detectors)]

# --- Drop Specific Source-Detector Pairs ---
df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
df1 = df1[~((df1['source'] == 16) & (df1['detector'] == 12))]

df2 = df2[~((df2['source'] == 14) & (df2['detector'] == 13))]
df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Only Those Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Group-level Pearson correlation across subjects ---
correlation_results = []

for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]
    
    for (src, det), chan_data in cond_data.groupby(['source', 'detector']):
        session1 = chan_data[chan_data['session'] == 1].sort_values('subject')
        session2 = chan_data[chan_data['session'] == 2].sort_values('subject')

        # Match subjects
        common_subs = set(session1['subject']).intersection(set(session2['subject']))
        session1 = session1[session1['subject'].isin(common_subs)].sort_values('subject')
        session2 = session2[session2['subject'].isin(common_subs)].sort_values('subject')

        if len(session1) > 1:
            try:
                r, p = pearsonr(session1['beta'].values, session2['beta'].values)
                correlation_results.append({
                    'condition': condition,
                    'channel': f"S{src}_D{det}",
                    'pearson_r': r,
                    'p_value': p,
                    'n_subjects': len(session1)
                })
            except Exception as e:
                print(f"❌ Error computing Pearson r for {condition}, S{src}_D{det}: {e}")

# --- Final Output ---
cor_df = pd.DataFrame(correlation_results)
print(cor_df)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# --- Step 1: Add significance flag and color ---
cor_df['significant'] = cor_df['p_value'] < 0.05
cor_df['bar_color'] = cor_df['significant'].map({True: 'seagreen', False: 'gray'})

# --- Step 2: Bootstrap CI for each (channel, condition) pair ---
boot_ci = []

for i, row in cor_df.iterrows():
    cond = row['condition']
    chan = row['channel']

    # Get matched data for this condition/channel
    cond_data = combined[
        (combined['cond'] == cond) &
        (combined['source'] == int(chan.split('_')[0][1:])) &
        (combined['detector'] == int(chan.split('_')[1][1:]))
    ]

    session1 = cond_data[cond_data['session'] == 1].sort_values('subject')
    session2 = cond_data[cond_data['session'] == 2].sort_values('subject')

    common_subjects = set(session1['subject']).intersection(set(session2['subject']))
    session1 = session1[session1['subject'].isin(common_subjects)].sort_values('subject')
    session2 = session2[session2['subject'].isin(common_subjects)].sort_values('subject')

    r_boot = []
    if len(session1) > 3:
        for _ in range(1000):
            idx = np.random.choice(len(session1), size=len(session1), replace=True)
            try:
                r, _ = pearsonr(session1['tstat'].values[idx], session2['tstat'].values[idx])
                r_boot.append(r)
            except:
                continue

        lower = np.percentile(r_boot, 2.5)
        upper = np.percentile(r_boot, 97.5)
    else:
        lower, upper = np.nan, np.nan

    boot_ci.append((lower, upper))

cor_df['ci_lower'], cor_df['ci_upper'] = zip(*boot_ci)
cor_df['yerr_lower'] = cor_df['pearson_r'] - cor_df['ci_lower']
cor_df['yerr_upper'] = cor_df['ci_upper'] - cor_df['pearson_r']

# --- Step 3: Plot with CIs and color coding ---
plt.figure(figsize=(14, 6))

# Plot each condition separately for better grouping
for condition in cor_df['condition'].unique():
    subset = cor_df[cor_df['condition'] == condition]
    x_pos = np.arange(len(subset))
    bar = plt.bar(
        x=x_pos,
        height=subset['pearson_r'],
        yerr=[subset['yerr_lower'], subset['yerr_upper']],
        capsize=4,
        color=subset['bar_color'],
        edgecolor='black',
        label=f'Condition {condition}'
    )

    for i, (x, r) in enumerate(zip(x_pos, subset['pearson_r'])):
        plt.text(x, r + 0.02, f"{r:.2f}", ha='center', va='bottom', fontsize=9)

    # Fix x-ticks
    plt.xticks(ticks=x_pos, labels=subset['channel'], rotation=90)

plt.axhline(0, color='gray', linestyle='--')
plt.ylabel('Pearson r')
plt.title('Subject-Level Session-to-Session Correlation by Channel and Condition')
plt.ylim(-0.2, 1.1)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# --- Compute Average Pearson r for Each Condition ---
average_corr = (
    cor_df
    .groupby('condition')['pearson_r']
    .mean()
    .reset_index()
    .rename(columns={'pearson_r': 'avg_pearson_r'})
)

print("📊 Average Pearson r by Condition:")
print(average_corr)




✅ Found 26 subjects in both sessions.
   condition  channel  pearson_r   p_value  n_subjects
0      x1-x2    S1_D1  -0.430141  0.035903          24
1      x1-x2    S1_D2  -0.158608  0.448900          25
2      x1-x2    S2_D1  -0.126038  0.576229          22
3      x1-x2    S2_D2  -0.137614  0.511840          25
4      x1-x2    S2_D3   0.098854  0.832998           7
5      x1-x2    S3_D1   0.104516  0.670241          19
6      x1-x2    S3_D3  -0.337589  0.134488          21
7      x1-x2    S4_D2  -0.224522  0.280593          25
8      x1-x2    S4_D3  -0.076202  0.723415          24
9      x1-x2    S4_D4   0.241811  0.349761          17
10     x1-x2    S5_D3  -0.045221  0.845675          21
11     x1-x2    S5_D4  -0.388952  0.110651          18
12     x1-x2    S6_D5   0.026529  0.897643          26
13     x1-x2    S6_D6  -0.141213  0.500755          25
14     x1-x2    S7_D5  -0.066685  0.768109          22
15     x1-x2    S7_D6  -0.208375  0.377987          20
16     x1-x2    S7_D7  -0.1

In [113]:
print(min(cor_df['pearson_r']))
print(max(cor_df['pearson_r'])) 
print(cor_df['pearson_r'].median())
print(cor_df['pearson_r'].std()) 

-0.4301413602679208
0.6459312289701848
0.08638121178175978
0.25065517648452934


In [119]:
## TASK VS REST - Channel Level Variance Correlation Code - SUBJECT LEVEL 

import pandas as pd
pd.set_option('display.max_rows', None)
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# # --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_glm_result_session1_tvr_clean.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_glm_result_session2_tvr_clean.csv"

# # --- Load Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session1_clean.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_subject_level_contrasts_session2_clean.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channels Only ---
df1 = df1[df1['type'].str.lower() == 'hbo'].copy()
df2 = df2[df2['type'].str.lower() == 'hbo'].copy()

# --- Normalize subject names by keeping only the last 7 characters ---
df1['subject'] = df1['subject'].astype(str).str[-7:]
df2['subject'] = df2['subject'].astype(str).str[-7:]

# --- Drop rows with NaN in 'tstat' column ---
df1 = df1.dropna(subset=['beta'])
df2 = df2.dropna(subset=['beta'])

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['detector'].isin(bad_detectors)]
df2 = df2[~df2['detector'].isin(bad_detectors)]

# --- Drop Specific Source-Detector Pairs ---
df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
df1 = df1[~((df1['source'] == 16) & (df1['detector'] == 12))]

df2 = df2[~((df2['source'] == 14) & (df2['detector'] == 13))]
df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Group-level Variance Correlation across subjects ---
variance_results = []

for condition in combined['cond'].unique():
    cond_data = combined[combined['cond'] == condition]
    
    for (src, det), chan_data in cond_data.groupby(['source', 'detector']):
        session1 = chan_data[chan_data['session'] == 1].sort_values('subject')
        session2 = chan_data[chan_data['session'] == 2].sort_values('subject')

        common_subs = set(session1['subject']).intersection(set(session2['subject']))
        session1 = session1[session1['subject'].isin(common_subs)].sort_values('subject')
        session2 = session2[session2['subject'].isin(common_subs)].sort_values('subject')

        if len(session1) > 1:
            try:
                var1 = session1['beta'].values ** 2
                var2 = session2['beta'].values ** 2
                r, p = pearsonr(var1, var2)
                variance_results.append({
                    'condition': condition,
                    'channel': f"S{src}_D{det}",
                    'pearson_r_variance': r,
                    'p_value': p,
                    'n_subjects': len(session1)
                })
            except Exception as e:
                print(f"❌ Error computing variance correlation for {condition}, S{src}_D{det}: {e}")

# --- Final Output ---
var_df = pd.DataFrame(variance_results)
print(var_df)
# --- Add significance flag and color ---
var_df['significant'] = var_df['p_value'] < 0.05
var_df['bar_color'] = var_df['significant'].map({True: 'seagreen', False: 'gray'})

# --- Plot with CIs and color coding ---
plt.figure(figsize=(14, 6))

for condition in var_df['condition'].unique():
    subset = var_df[var_df['condition'] == condition]
    x_pos = np.arange(len(subset))

    bar = plt.bar(
        x=x_pos,
        height=subset['pearson_r_variance'],
        color=subset['bar_color'],
        edgecolor='black',
        label=f'Condition {condition}'
    )

    for i, (x, r) in enumerate(zip(x_pos, subset['pearson_r_variance'])):
        plt.text(x, r + 0.02, f"{r:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(ticks=x_pos, labels=subset['channel'], rotation=90)

plt.axhline(0, color='gray', linestyle='--')
plt.ylabel('Pearson r (t²)')
plt.title('Subject-Level Session-to-Session Variance Correlation by Channel and Condition')
plt.ylim(-0.2, 1.1)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# --- Compute Average Variance Correlation for Each Condition ---
average_var_corr = (
    var_df
    .groupby('condition')['pearson_r_variance']
    .mean()
    .reset_index()
    .rename(columns={'pearson_r_variance': 'avg_pearson_r_variance'})
)

print("📊 Average Pearson r (t²) by Condition:")
print(average_var_corr)

   condition  channel  pearson_r_variance   p_value  n_subjects
0         x1    S1_D1            0.738682  0.000038          24
1         x1    S1_D2            0.320939  0.117758          25
2         x1    S2_D1            0.765306  0.000033          22
3         x1    S2_D2            0.519054  0.007844          25
4         x1    S2_D3           -0.229890  0.619957           7
5         x1    S3_D1            0.289527  0.229245          19
6         x1    S3_D3           -0.039756  0.864149          21
7         x1    S4_D2            0.427293  0.033134          25
8         x1    S4_D3            0.300264  0.153983          24
9         x1    S4_D4            0.292740  0.254176          17
10        x1    S5_D3            0.126527  0.584709          21
11        x1    S5_D4            0.255012  0.307128          18
12        x1    S6_D5            0.438874  0.024900          26
13        x1    S6_D6            0.210333  0.312893          25
14        x1    S7_D5            0.12912

In [121]:
print(min(var_df['pearson_r_variance']))
print(max(var_df['pearson_r_variance'])) 
print(var_df['pearson_r_variance'].median())
print(var_df['pearson_r_variance'].std()) 

-0.22989000375743296
0.8385210256438642
0.281027212273254
0.27858406654949347


In [29]:
## Group Level Functional Connectivity Correlations

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# # --- Load Group-Level Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_group_functional_conn_tvr_session1.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_group_functional_conn_tvr_session2.csv"

# --- Load Group-Level Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\GLOBAL_26_group_functional_conn_tvr_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\GLOBAL_26_group_functional_conn_tvr_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['DetectorOrigin'].isin(bad_detectors)]
df2 = df2[~df2['DetectorOrigin'].isin(bad_detectors)]
df1 = df1[~df1['DetectorDest'].isin(bad_detectors)]
df2 = df2[~df2['DetectorDest'].isin(bad_detectors)]

# --- Remove mirror-flipped channel pairs ---
def make_pair_id(df):
    return df.apply(
        lambda row: tuple(sorted([
            (row['SourceOrigin'], row['DetectorOrigin']),
            (row['SourceDest'], row['DetectorDest'])
        ])), axis=1)

df1['pair_id'] = make_pair_id(df1)
df2['pair_id'] = make_pair_id(df2)

# Keep only one direction per pair
df1 = df1.drop_duplicates(subset='pair_id')
df2 = df2.drop_duplicates(subset='pair_id')

# Drop the helper column
df1 = df1.drop(columns='pair_id')
df2 = df2.drop(columns='pair_id')

# --- Add Session Labels ---
df1['session'] = 1
df2['session'] = 2

# --- Merge on Channel Pair Info ---
merged = pd.merge(
    df1, df2,
    on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
    suffixes=('_s1', '_s2')
).dropna(subset=['R_s1', 'R_s2'])

# --- Compute Group-Level Correlation of R Values ---
r_val, p_val = pearsonr(merged['R_s1'], merged['R_s2'])

print(f"✅ Group-level R correlation across sessions: r = {r_val:.4f}, p = {p_val:.20g}")
print(f"Number of matched channel pairs: {len(merged)}")

# --- Paired t-test ---
t_stat, t_p = ttest_rel(merged['R_s1'], merged['R_s2'])
print(f"📊 Paired t-test result: t = {t_stat:.4f}, p = {t_p:.4f}")

# --- Mean difference ---
diff = merged['R_s2'] - merged['R_s1']
mean_diff = diff.mean()
print(f"📉 Mean difference (R_s2 - R_s1): {mean_diff:.4f}")

# --- Plot correlation scatter ---
plt.figure(figsize=(6, 4))
sns.scatterplot(x=merged['R_s1'], y=merged['R_s2'], s=60, color='red', edgecolor='black')
plt.plot(
    [merged['R_s1'].min(), merged['R_s1'].max()],
    [merged['R_s1'].min(), merged['R_s1'].max()],
    linestyle='--', color='black'
)
plt.title(f"Session-to-Session Functional Connectivity \nr = {r_val:.2f}", fontsize=12)
plt.xlabel("r (Session 1)")
plt.ylabel("r (Session 2)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- Plot difference score distribution ---
plt.figure(figsize=(6, 4))
sns.histplot(diff, bins=30, kde=True, color='steelblue')
plt.axvline(0, color='black', linestyle='--')
plt.title("Distribution of Differences in FC (Session 2 - Session 1)", fontsize=12)
plt.xlabel("Difference in r values")
plt.ylabel("Count")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_group_correlation_FunctionalConnectivity_tvr.png"  # <-- EDIT THIS PATH
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")

plt.show()


✅ Group-level R correlation across sessions: r = 0.9856, p = 0
Number of matched channel pairs: 741
📊 Paired t-test result: t = 0.3657, p = 0.7147
📉 Mean difference (R_s2 - R_s1): -0.0005


In [31]:
## Functional Connectivity correlation - Subject level - TASK VS REST 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_tvr_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_tvr_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['DetectorOrigin'].isin(bad_detectors)]
df2 = df2[~df2['DetectorOrigin'].isin(bad_detectors)]
df1 = df1[~df1['DetectorDest'].isin(bad_detectors)]
df2 = df2[~df2['DetectorDest'].isin(bad_detectors)]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Functional Connectivity Pattern for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Merge on the full connection pair (origin + destination)
    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched pairs for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['R_s1'], merged['R_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_pairs': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)
print(np.mean(subject_corr_df['pearson_r']))

# --- Plot Pearson r Values for Each Subject ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['pearson_r'] + 0.02,
        f"{row['pearson_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Correlation of Functional Connectivity', fontsize=18)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_Subject_correlation_FunctionalConnectivity_tvr.png"  # <-- EDIT THIS PATH
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")

plt.show()

✅ Found 26 subjects in both sessions.
   subject  pearson_r  n_pairs
0   pid001   0.716014     1444
1   pid002   0.798743     1444
2   pid003   0.658569     1444
3   pid004   0.850785     1444
4   pid005   0.568208     1444
5   pid006   0.546049     1444
6   pid007   0.759268     1444
7   pid008   0.786351     1444
8   pid009   0.569396     1444
9   pid010   0.734643     1444
10  pid011   0.722093     1444
11  pid012   0.502109     1444
12  pid013   0.720404     1444
13  pid014   0.854016     1444
14  pid015   0.746654     1444
15  pid016   0.716004     1444
16  pid017   0.949136     1444
17  pid018   0.230852     1444
18  pid019   0.660373     1444
19  pid020   0.748112     1444
20  pid021   0.826523     1444
21  pid022   0.718691     1444
22  pid023   0.678993     1444
23  pid024   0.685745     1444
24  pid025   0.499203     1444
25  pid026   0.862859     1444
0.6965305437213927


C:\Users\pkell\AppData\Local\Temp\ipykernel_15876\819613838.py:79: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [17]:
## Functional Connectivity correlation - Subject level - TASK VS REST 

import pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np  # (needed for means and ICC math)

# # --- Load Data ---
# file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_tvr_session1.csv"
# file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_tvr_session2.csv"

# --- Load ROI Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_ROI_AGG_tvr_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_ROI_AGG_tvr_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['DetectorOrigin'].isin(bad_detectors)]
df2 = df2[~df2['DetectorOrigin'].isin(bad_detectors)]
df1 = df1[~df1['DetectorDest'].isin(bad_detectors)]
df2 = df2[~df2['DetectorDest'].isin(bad_detectors)]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Pearson r for Functional Connectivity Pattern for Each Subject ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]

    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    # Merge on the full connection pair (origin + destination)
    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        print(f"⚠️ Not enough matched pairs for {subject}. Skipping.")
        continue

    try:
        r, p = pearsonr(merged['R_s1'], merged['R_s2'])
        participant_results.append({
            'subject': subject,
            'pearson_r': r,
            'n_pairs': merged.shape[0]
        })
    except Exception as e:
        print(f"❌ Error for {subject}: {e}")

# --- Results DataFrame ---
subject_corr_df = pd.DataFrame(participant_results)
print(subject_corr_df)
print(np.mean(subject_corr_df['pearson_r']))

# --- Plot Pearson r Values for Each Subject ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')

# Add text labels above each bar
for i, row in subject_corr_df.iterrows():
    ax.text(
        i,
        row['pearson_r'] + 0.02,
        f"{row['pearson_r']:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Participant-Level Correlation of Functional Connectivity', fontsize=18)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\26_Subject_correlation_FunctionalConnectivity_tvr.png"  # <-- EDIT THIS PATH
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")

plt.show()

# ======================================================================
# === ADDED CODE (updated): Fisher-z transform + Edgewise ICC(2,1) + Subject-level ICC(2,1) + Plots
# ======================================================================

# -----------------------------
# Prepare long data with edge IDs (channel pair)
# -----------------------------
combined = combined.copy()
combined['edge_id'] = (
    combined['SourceOrigin'].astype(str) + '_' +
    combined['DetectorOrigin'].astype(str) + '__' +
    combined['SourceDest'].astype(str) + '_' +
    combined['DetectorDest'].astype(str)
)

# --- Fisher z-transform of R for stability (clip to avoid +/-inf) ---
eps = 1e-7
combined['R_clipped'] = combined['R'].clip(-1 + eps, 1 - eps)
combined['Z'] = np.arctanh(combined['R_clipped'])

# -----------------------------
# EDGEWISE ICC(2,1) across subjects (targets = subjects, raters = sessions)
# -----------------------------
pivot_edges = combined.pivot_table(
    index=['subject', 'edge_id'],
    columns='session',
    values='Z',
    aggfunc='mean'  # in case of duplicates
).dropna(subset=[1, 2])

def icc_two_way_ms(Y):
    """
    Two-way ANOVA mean squares for subjects (n) and raters (k).
    Y: array shape (n_targets, k_raters)
    Returns: MS_subjects, MS_raters, MS_error, n, k
    """
    Y = np.asarray(Y, dtype=float)
    n, k = Y.shape
    grand = Y.mean()
    subj_means = Y.mean(axis=1)
    rater_means = Y.mean(axis=0)

    ss_subj = k * np.sum((subj_means - grand) ** 2)
    ss_rater = n * np.sum((rater_means - grand) ** 2)
    ss_total = np.sum((Y - grand) ** 2)
    ss_error = ss_total - ss_subj - ss_rater

    ms_subj = ss_subj / (n - 1) if n > 1 else np.nan
    ms_rater = ss_rater / (k - 1) if k > 1 else np.nan
    ms_error = ss_error / ((n - 1) * (k - 1)) if (n > 1 and k > 1) else np.nan
    return ms_subj, ms_rater, ms_error, n, k

def icc_2_1(Y):
    """
    ICC(2,1): Two-way random effects, absolute agreement.
    (MS_S - MS_E) / (MS_S + (k-1)MS_E + (k*(MS_R - MS_E)/n))
    """
    ms_s, ms_r, ms_e, n, k = icc_two_way_ms(Y)
    if any(np.isnan([ms_s, ms_r, ms_e])) or n < 2 or k < 2:
        return np.nan
    return (ms_s - ms_e) / (ms_s + (k - 1) * ms_e + (k * (ms_r - ms_e) / n))

# -- Compute EDGEWISE ICCs
icc_rows = []
for edge_id, subdf in pivot_edges.groupby(level='edge_id'):
    Y = subdf[[1, 2]].to_numpy()  # rows=subjects, cols=sessions
    if Y.shape[0] < 3:
        continue
    icc2 = icc_2_1(Y)
    icc_rows.append({'edge_id': edge_id, 'n_subjects': Y.shape[0], 'icc_2_1': icc2})

icc_df = pd.DataFrame(icc_rows).dropna(subset=['icc_2_1'])
print(f"Computed EDGEWISE ICC(2,1) for {len(icc_df)} edges (min subjects per edge >= 3).")
print(icc_df[['icc_2_1']].describe())

# --- Plot: Distribution of EDGEWISE ICC(2,1)
plt.figure(figsize=(7, 5))
sns.histplot(icc_df['icc_2_1'], bins=30, kde=True, edgecolor='black')
plt.title('Edgewise ICC(2,1) — Absolute Agreement')
plt.xlabel('ICC(2,1)')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# --- Summary thresholds
median_21 = icc_df['icc_2_1'].median()
pct_mod = (icc_df['icc_2_1'] >= 0.40).mean() * 100.0
pct_good = (icc_df['icc_2_1'] >= 0.60).mean() * 100.0
print(f"Median EDGEWISE ICC(2,1): {median_21:.3f} | % ≥ .40: {pct_mod:.1f}% | % ≥ .60: {pct_good:.1f}%")

# ======================================================================
# === ADDED PLOTS: Channel×Channel heatmap and ranked lollipop of edge ICCs
# ======================================================================
import re

# --- Build mapping: edge_id -> (chOrigin, chDest) as "S#_D#" labels ---
def edge_to_channels(edge_id):
    # edge_id format: "<SrcO>_<DetO>__<SrcD>_<DetD>"
    m = re.match(r'^(\d+)\_(\d+)__\s*(\d+)\_(\d+)$', edge_id)
    if m is None:
        left, right = edge_id.split('__')
        so, do = left.split('_')
        sd, dd = right.split('_')
        return f"S{so}_D{do}", f"S{sd}_D{dd}"
    so, do, sd, dd = map(int, m.groups())
    return f"S{so}_D{do}", f"S{sd}_D{dd}"

edge_map = icc_df[['edge_id', 'icc_2_1']].copy()
edge_map[['chOrigin', 'chDest']] = edge_map['edge_id'].apply(lambda e: pd.Series(edge_to_channels(e)))

# --- Channel order (numeric by S then D), include origins and any dest-only channels ---
chan_o = pd.DataFrame({'S': combined['SourceOrigin'].astype(int), 'D': combined['DetectorOrigin'].astype(int)}).drop_duplicates()
chan_o = chan_o.sort_values(['S', 'D'])
chan_order = [f"S{r.S}_D{r.D}" for r in chan_o.itertuples(index=False)]
chan_d = pd.DataFrame({'S': combined['SourceDest'].astype(int), 'D': combined['DetectorDest'].astype(int)}).drop_duplicates()
extra = set([f"S{r.S}_D{r.D}" for r in chan_d.itertuples(index=False)]) - set(chan_order)
chan_order.extend(sorted(extra, key=lambda x: (int(x.split('_')[0][1:]), int(x.split('_')[1][1:]))))

# --- Symmetric ICC matrix (channels × channels) ---
icc_mat = pd.DataFrame(np.nan, index=chan_order, columns=chan_order, dtype=float)
for row in edge_map.itertuples(index=False):
    i = row.chOrigin
    j = row.chDest
    v = row.icc_2_1
    if i in icc_mat.index and j in icc_mat.columns:
        icc_mat.loc[i, j] = v
        icc_mat.loc[j, i] = v
for c in icc_mat.index:
    icc_mat.loc[c, c] = np.nan  # no self-connections

# --- Heatmap (lower triangle) ---
mask = np.triu(np.ones_like(icc_mat, dtype=bool))
plt.figure(figsize=(12, 10))
ax = sns.heatmap(
    icc_mat,
    mask=mask,
    cmap='viridis',
    vmin=-0.2, vmax=1.0,
    linewidths=0.2, linecolor='white',
    cbar_kws={'label': 'ICC(2,1)'}
)
plt.title('Edgewise Between-Session ICC(2,1) — Channel × Channel', fontsize=16)
plt.xlabel('Channel (Dest)')
plt.ylabel('Channel (Origin)')
plt.tight_layout()
plt.show()

# --- Ranked edge ICC lollipop plot ---
ranked_edges = edge_map.copy()
ranked_edges['pair'] = ranked_edges['chOrigin'] + ' ↔ ' + ranked_edges['chDest']
ranked_edges = ranked_edges.sort_values('icc_2_1', ascending=True)

plt.figure(figsize=(10, 12))
ypos = np.arange(len(ranked_edges))
plt.hlines(y=ypos, xmin=0, xmax=ranked_edges['icc_2_1'], alpha=0.4)
plt.plot(ranked_edges['icc_2_1'], ypos, 'o', markersize=4)
plt.yticks(ypos, ranked_edges['pair'])
plt.xlabel('ICC(2,1)')
plt.title('Edgewise ICC(2,1) — Ranked Channel Pairs')
plt.xlim((-0.2, 1.0))
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# -----------------------------
# SUBJECT-LEVEL ICC(2,1): reliability per subject using edges as targets
#   (targets = edges within that subject; raters = sessions)
# -----------------------------
subject_icc_rows = []
for subject in sorted(combined['subject'].unique()):
    sub = combined[combined['subject'] == subject]
    # Build per-subject pivot: rows = edges, cols = sessions (Z values)
    sub_pivot = sub.pivot_table(
        index='edge_id',
        columns='session',
        values='Z',
        aggfunc='mean'
    ).dropna(subset=[1, 2])
    Y = sub_pivot[[1, 2]].to_numpy()
    if Y.shape[0] < 3:
        subject_icc_rows.append({'subject': subject, 'icc_2_1_subject': np.nan, 'n_edges': Y.shape[0]})
        continue
    icc2_subj = icc_2_1(Y)
    subject_icc_rows.append({'subject': subject, 'icc_2_1_subject': icc2_subj, 'n_edges': Y.shape[0]})

subject_icc_df = pd.DataFrame(subject_icc_rows)
print("Subject-level ICC(2,1) summary:")
print(subject_icc_df.describe())

# -----------------------------
# Plot: SUBJECT-LEVEL ICC(2,1) as a dot plot (ordered pid001..pid026)
# -----------------------------
# Build desired order and keep those subjects that actually exist
desired_order = [f"pid{str(i).zfill(3)}" for i in range(1, 27)]
present_order = [s for s in desired_order if s in subject_icc_df['subject'].unique().tolist()]

# Ensure categorical ordering for plotting
subject_icc_df = subject_icc_df.copy()
subject_icc_df['subject'] = pd.Categorical(subject_icc_df['subject'], categories=present_order, ordered=True)
subject_icc_df = subject_icc_df.sort_values('subject')

plt.figure(figsize=(10, 5))
sns.stripplot(
    data=subject_icc_df,
    x='subject',
    y='icc_2_1_subject',
    size=8,
    jitter=False,
    edgecolor='black',
    linewidth=0.5
)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Subject (ordered pid001 → pid026)')
plt.ylabel('Subject-level ICC(2,1) (edges as targets)')
plt.title('Between-Session Reliability per Subject (ICC(2,1))')
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# ======================================================================
# === NEW: Node-strength ICC (mean ICC across all edges touching a node)
# ======================================================================

# Compute per-channel mean ICC by averaging each row (or column) of icc_mat
node_icc_mean = icc_mat.mean(axis=1, skipna=True)
node_icc_count = icc_mat.notna().sum(axis=1)

node_icc_df = (
    pd.DataFrame({
        'channel': node_icc_mean.index,
        'node_strength_icc': node_icc_mean.values,
        'n_edges_contributing': node_icc_count.values
    })
    .sort_values('node_strength_icc')
    .reset_index(drop=True)
)

print("Node-strength ICC summary (mean edge ICC per channel):")
print(node_icc_df.describe())

# --- Plot: Node-strength ICC (sorted ascending to flag low-reliability channels) ---
plt.figure(figsize=(12, 6))
sns.barplot(
    data=node_icc_df,
    x='channel',
    y='node_strength_icc',
    color='tab:blue',
    edgecolor='black'
)
plt.xticks(rotation=90)
plt.ylabel('Mean ICC(2,1) of incident edges')
plt.xlabel('Channel')
plt.title('Node-Strength ICC per Channel (lower = less reliable)')
# Threshold guides (optional)
plt.axhline(0.40, ls='--', color='gray', alpha=0.7)
plt.axhline(0.60, ls='--', color='gray', alpha=0.7)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# --- (Optional) Print a short table of the lowest and highest channels ---
k = 10
print("\nLowest node-strength ICC channels:")
print(node_icc_df.head(k))
print("\nHighest node-strength ICC channels:")
print(node_icc_df.tail(k).iloc[::-1])

# ======================================================================
# === NEW: Per-subject MAE / RMSE between sessions (Fisher-z edges)
# ======================================================================

mae_rows = []
for subject in present_order:  # use same subject ordering
    if subject not in combined['subject'].unique():
        continue
    sub = combined[combined['subject'] == subject].copy()
    sub_pivot = sub.pivot_table(index='edge_id', columns='session', values='Z', aggfunc='mean').dropna(subset=[1, 2])
    n_edges = sub_pivot.shape[0]
    if n_edges < 3:
        mae_rows.append({'subject': subject, 'n_edges': n_edges, 'MAE': np.nan, 'RMSE': np.nan})
        continue
    diff = (sub_pivot[2] - sub_pivot[1]).to_numpy()
    mae = np.mean(np.abs(diff))
    rmse = np.sqrt(np.mean(diff**2))
    mae_rows.append({'subject': subject, 'n_edges': n_edges, 'MAE': mae, 'RMSE': rmse})

mae_df = pd.DataFrame(mae_rows)
print("\nPer-subject MAE/RMSE (Fisher-z units):")
print(mae_df.describe())

# Plot MAE per subject
plt.figure(figsize=(10, 4))
sns.barplot(data=mae_df, x='subject', y='MAE', color='tab:blue', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.ylabel('MAE (|z_S2 - z_S1|)')
plt.xlabel('Subject')
plt.title('Per-subject Mean Absolute Error between Sessions (Fisher-z)')
plt.tight_layout()
plt.show()

# Relate MAE to pattern correlation r
tmp = pd.merge(subject_corr_df[['subject', 'pearson_r']], mae_df, on='subject', how='inner')
plt.figure(figsize=(5, 5))
sns.scatterplot(data=tmp, x='MAE', y='pearson_r')
plt.gca().invert_xaxis()  # typically lower MAE -> higher r
plt.xlabel('MAE (Fisher-z)')
plt.ylabel('Vector correlation (r)')
plt.title('Pattern Similarity vs Magnitude Error')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.
   subject  pearson_r  n_pairs
0   pid001   0.843653      100
1   pid002   0.913117      100
2   pid003   0.916637      100
3   pid004   0.822661      100
4   pid005   0.910607      100
5   pid006   0.934384      100
6   pid007   0.957217      100
7   pid008   0.953805      100
8   pid009   0.936296      100
9   pid010   0.942815      100
10  pid011   0.932412      100
11  pid012   0.884262      100
12  pid013   0.874480      100
13  pid014   0.960001      100
14  pid015   0.946386      100
15  pid016   0.671583      100
16  pid017   0.980907      100
17  pid018   0.490508      100
18  pid019   0.971938      100
19  pid020   0.940483      100
20  pid021   0.957959      100
21  pid022   0.948413      100
22  pid023   0.923248      100
23  pid024   0.968595      100
24  pid025   0.946068      100
25  pid026   0.979158      100
0.9041381900190916
Computed EDGEWISE ICC(2,1) for 100 edges (min subjects per edge >= 3).
          icc_2_1
count  100.000000

C:\Users\pkell\anaconda3\Lib\site-packages\seaborn\categorical.py:253: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)


Node-strength ICC summary (mean edge ICC per channel):
       node_strength_icc  n_edges_contributing
count          10.000000                  10.0
mean            0.536975                   9.0
std             0.137572                   0.0
min             0.231003                   9.0
25%             0.493052                   9.0
50%             0.541238                   9.0
75%             0.574255                   9.0
max             0.730486                   9.0

Lowest node-strength ICC channels:
   channel  node_strength_icc  n_edges_contributing
0    S4_D3           0.231003                     9
1    S8_D7           0.472472                     9
2    S9_D6           0.482347                     9
3    S7_D5           0.525164                     9
4   S11_D9           0.535263                     9
5    S2_D2           0.547212                     9
6  S15_D14           0.556542                     9
7    S3_D1           0.580159                     9
8  S14_D13        

In [65]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# --- Create DataFrame from results ---
subject_corr_df = pd.DataFrame(participant_results)

# --- Extract data ---
data = subject_corr_df['pearson_r'].dropna()
x_center = 0  # central x position

# --- Set up plot ---
plt.figure(figsize=(8, 6))

# 1. Half Violin (Right Side Only)
kde = gaussian_kde(data, bw_method=0.3)
y_vals = np.linspace(data.min(), data.max(), 300)
densities = kde(y_vals)
densities = densities / densities.max() * 0.4  # scale width
plt.fill_betweenx(y_vals, x_center, x_center + densities, facecolor='lightblue', alpha=0.7)

# 2. Stripplot Dots (Left Side)
# We'll manually scatter them with jitter
jitter = np.random.uniform(-0.05, 0.05, size=len(data))
plt.scatter(np.full_like(data, x_center - 0.15) + jitter, data, color='black', alpha=0.7, edgecolor='white', linewidth=0.5)

# 3. Boxplot (Centered)
# Compute boxplot stats manually to plot at x=0
box_data = data.values
median = np.median(box_data)
q1 = np.percentile(box_data, 25)
q3 = np.percentile(box_data, 75)
iqr = q3 - q1
whisker_low = np.min(box_data[box_data >= q1 - 1.5 * iqr])
whisker_high = np.max(box_data[box_data <= q3 + 1.5 * iqr])

# Draw box
plt.plot([x_center - 0.05, x_center + 0.05], [q1, q1], color='black')  # bottom
plt.plot([x_center - 0.05, x_center + 0.05], [q3, q3], color='black')  # top
plt.plot([x_center - 0.05, x_center - 0.05], [q1, q3], color='black')  # left
plt.plot([x_center + 0.05, x_center + 0.05], [q1, q3], color='black')  # right
plt.plot([x_center - 0.05, x_center + 0.05], [median, median], color='red', linewidth=2)  # median line

# Whiskers
plt.plot([x_center, x_center], [q3, whisker_high], color='black', linestyle='-')
plt.plot([x_center, x_center], [q1, whisker_low], color='black', linestyle='-')

# Caps
plt.plot([x_center - 0.02, x_center + 0.02], [whisker_high, whisker_high], color='black')
plt.plot([x_center - 0.02, x_center + 0.02], [whisker_low, whisker_low], color='black')

# --- Final styling ---
plt.xlim(-0.5, 0.6)
plt.ylim(0, 1.05)
plt.xticks([])
plt.ylabel("Pearson r (Session 1 vs Session 2)")
plt.title("Raincloud Plot of Participant-Level FC Correlation", fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

# --- Show ---
plt.show()


In [21]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load Data ---
file1 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_tvr_session1.csv"
file2 = r"C:\Users\pkell\OneDrive\Documents\MATLAB\26_functional_conn_subjectlevel_tvr_session2.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# --- Filter to HbO Channel Pairs Only ---
df1 = df1[(df1['TypeOrigin'].str.lower() == 'hbo') & (df1['TypeDest'].str.lower() == 'hbo')].copy()
df2 = df2[(df2['TypeOrigin'].str.lower() == 'hbo') & (df2['TypeDest'].str.lower() == 'hbo')].copy()

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['DetectorOrigin'].isin(bad_detectors)]
df2 = df2[~df2['DetectorOrigin'].isin(bad_detectors)]
df1 = df1[~df1['DetectorDest'].isin(bad_detectors)]
df2 = df2[~df2['DetectorDest'].isin(bad_detectors)]

# --- Add Session Identifier ---
df1['session'] = 1
df2['session'] = 2

# --- Find Subjects Present in Both Sessions ---
common_subjects = set(df1['subject']).intersection(set(df2['subject']))
print(f"✅ Found {len(common_subjects)} subjects in both sessions.")

# --- Filter to Common Subjects ---
df1 = df1[df1['subject'].isin(common_subjects)]
df2 = df2[df2['subject'].isin(common_subjects)]

# --- Combine Both Sessions ---
combined = pd.concat([df1, df2], ignore_index=True)

# --- Function to Remove Mirror-Flip Channel Pairs ---
def remove_mirror_pairs(df):
    df = df.copy()
    df['pair_id'] = df.apply(lambda row: tuple(sorted([
        (row['SourceOrigin'], row['DetectorOrigin']),
        (row['SourceDest'], row['DetectorDest'])
    ])), axis=1)
    df = df.drop_duplicates(subset=['subject', 'pair_id'])
    return df.drop(columns='pair_id')

# --- Remove Mirror Pairs ---
df1 = remove_mirror_pairs(df1)
df2 = remove_mirror_pairs(df2)
combined = pd.concat([df1, df2], ignore_index=True)

# --- Compute Per-Subject Pearson r and Global FC Values ---
participant_results = []

for subject in combined['subject'].unique():
    subj_data = combined[combined['subject'] == subject]
    session1 = subj_data[subj_data['session'] == 1]
    session2 = subj_data[subj_data['session'] == 2]

    merged = pd.merge(
        session1,
        session2,
        on=['SourceOrigin', 'DetectorOrigin', 'SourceDest', 'DetectorDest'],
        suffixes=('_s1', '_s2')
    )

    if merged.shape[0] < 3:
        continue

    r, _ = pearsonr(merged['R_s1'], merged['R_s2'])
    global_r_s1 = merged['R_s1'].mean()
    global_r_s2 = merged['R_s2'].mean()

    participant_results.append({
        'subject': subject,
        'pearson_r': r,
        'global_r_s1': global_r_s1,
        'global_r_s2': global_r_s2,
        'n_pairs': merged.shape[0]
    })

subject_corr_df = pd.DataFrame(participant_results)

# --- Create Separate Global FC DataFrames ---
global_fc_session1_df = subject_corr_df[['subject', 'global_r_s1']].rename(columns={'global_r_s1': 'global_r'})
global_fc_session2_df = subject_corr_df[['subject', 'global_r_s2']].rename(columns={'global_r_s2': 'global_r'})

# --- Plot Subject-Level Pearson r ---
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')
for i, row in subject_corr_df.iterrows():
    ax.text(i, row['pearson_r'] + 0.02, f"{row['pearson_r']:.2f}", ha='center', va='bottom', fontsize=9)

plt.title('Participant-Level Correlation of Functional Connectivity', fontsize=18)
plt.xlabel('Subject')
plt.ylabel('Pearson r (Session 1 vs Session 2)')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(subject_corr_df['pearson_r']) + 0.1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- Group-Level Correlation Between Global FC Values Across Sessions ---
group_r, group_p = pearsonr(subject_corr_df['global_r_s1'], subject_corr_df['global_r_s2'])
print(f"\n📊 Group-level global FC correlation: r = {group_r:.3f}, p = {group_p:.4f}")

# --- Scatter Plot of Global FC Session 1 vs Session 2 ---
plt.figure(figsize=(6, 5))
sns.scatterplot(x='global_r_s1', y='global_r_s2', data=subject_corr_df, s=70, color='darkorange', edgecolor='black')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title(f'Global FC Across Sessions\nr = {group_r:.2f}, p = {group_p:.4f}')
plt.xlabel('Global r (Session 1)')
plt.ylabel('Global r (Session 2)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


✅ Found 26 subjects in both sessions.

📊 Group-level global FC correlation: r = 0.791, p = 0.0000


C:\Users\pkell\AppData\Local\Temp\ipykernel_15876\3643607198.py:94: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=subject_corr_df, x='subject', y='pearson_r', palette='viridis', edgecolor='black')


In [25]:
global_fc_session2_df

,subject,global_r
0,pid001,0.373647
1,pid002,0.336332
2,pid003,0.393453
3,pid004,0.137747
4,pid005,0.318769
5,pid006,0.287290
6,pid007,0.167746
7,pid008,0.384055
8,pid009,0.391456
9,pid010,0.376359
